In [1]:

!pip install -q -U transformers accelerate sentence-transformers faiss-cpu pypdf peft datasets


In [2]:
import torch
import numpy as np
import os
import gc

from google.colab import files
from pypdf import PdfReader
from sentence_transformers import SentenceTransformer
from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM,
    Trainer,
    TrainingArguments,
    DataCollatorForLanguageModeling
)
from datasets import Dataset
from peft import LoraConfig, TaskType, get_peft_model

print("GPU available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))


GPU available: False


In [3]:
MODEL_NAME = "Qwen/Qwen2.5-0.5B-Instruct"

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    torch_dtype=(
        torch.float16 if torch.cuda.is_available()
        else torch.float32
    ),
    device_map="auto"
)

model.eval()
print("Open-source LLM loaded successfully!")


config.json:   0%|          | 0.00/659 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/7.30k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/7.03M [00:00<?, ?B/s]

...

model.safetensors: reconstructing file:   0%|          |  0.00B /  988MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

Open-source LLM loaded successfully!


In [4]:
def generate_answer(question, context=""):
    messages = [
        {
            "role": "system",
            "content": (
                "You are a personalized AI assistant. "
                "Answer clearly and accurately. When document context "
                "is provided, use it as the main source. If the answer "
                "is not supported by the context, say that it was not "
                "found in the supplied documents."
            )
        },
        {
            "role": "user",
            "content": (
                f"Document context:\n{context}\n\n"
                f"Question: {question}"
            )
        }
    ]

    prompt = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True
    )

    inputs = tokenizer(
        prompt,
        return_tensors="pt",
        truncation=True,
        max_length=2048
    ).to(model.device)

    model.eval()

    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=180,
            do_sample=False,
            pad_token_id=tokenizer.pad_token_id
        )

    new_tokens = outputs[0][inputs["input_ids"].shape[1]:]

    return tokenizer.decode(
        new_tokens,
        skip_special_tokens=True
    )


print(generate_answer("What is Generative AI?"))


Generative AI refers to artificial intelligence (AI) systems designed to generate human-like content or responses based on input data. These systems can be trained using various types of inputs such as text, images, audio, or video. The goal of generative AI is to create intelligent agents that can produce new information or experiences based on previously learned patterns or rules.

Key characteristics of generative AI include:

1. **Reinforcement Learning**: This involves learning through trial and error, where an agent learns from its own actions and environment feedback.
2. **Natural Language Processing (NLP)**: The ability to understand, interpret, generate, and respond to natural language inputs.
3. **Machine Learning**: The process of training models to learn from data without being explicitly programmed.
4. **Deep Learning**: A subset of machine learning that uses neural networks with multiple layers to perform tasks like image recognition, speech


In [7]:
!apt-get update -qq
!apt-get install -y -qq tesseract-ocr
!pip install -q pymupdf pytesseract pillow


W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu noble InRelease' does not seem to provide it (sources.list entry misspelt?)


In [8]:
import fitz
import pytesseract
import io
from PIL import Image
from google.colab import files

uploaded_files = files.upload()
documents = []

for filename in uploaded_files:
    if filename.lower().endswith(".pdf"):
        pdf = fitz.open(filename)

        for page_num, page in enumerate(pdf, start=1):
            # First try normal text extraction
            text = page.get_text("text").strip()

            # If scanned PDF, use OCR
            if len(text) < 20:
                pix = page.get_pixmap(
                    matrix=fitz.Matrix(2, 2),
                    alpha=False
                )

                image = Image.open(
                    io.BytesIO(pix.tobytes("png"))
                )

                text = pytesseract.image_to_string(
                    image,
                    lang="eng"
                ).strip()

            if text:
                documents.append({
                    "text": text,
                    "source": filename,
                    "page": page_num
                })

        pdf.close()

    elif filename.lower().endswith(".txt"):
        with open(filename, "r", encoding="utf-8") as f:
            text = f.read().strip()

        if text:
            documents.append({
                "text": text,
                "source": filename,
                "page": 1
            })

print("Documents loaded:", len(documents))

if documents:
    print("\nExtracted text preview:\n")
    print(documents[0]["text"][:1000])
else:
    print("No text extracted. Check PDF quality or OCR language.")


/usr/local/lib/python3.13/dist-packages/fitz/__init__.py:17: FutureWarning: The `fitz` API is deprecated and will be removed in future. Use `import pymupdf` instead.
  warnings.warn(


Saving User Manual.pdf to User Manual.pdf
Documents loaded: 19

Extracted text preview:

802.11b/g/n Wireless USB Adapter 
 
1 
 
Content 
Chapter 1 Product Introduction .................................................................................................... 2 
1.1 Package Contents ................................................................................................................ 2 
1.2 Product Features .................................................................................................................. 2 
1.3 Product Usage ..................................................................................................................... 2 
1.4 Before Installation ............................................................................................................... 2 
1.5 Disable other manufacturers’ wireless network adapters .................................................... 3 
1.6 Product Maintenance .................................

In [9]:
# Split extracted documents into smaller chunks

chunk_size = 500
overlap = 100
chunks = []

for doc in documents:
    text = doc["text"]
    start = 0

    while start < len(text):
        end = start + chunk_size
        chunk_text = text[start:end].strip()

        if chunk_text:
            chunks.append({
                "text": chunk_text,
                "source": doc["source"],
                "page": doc["page"]
            })

        start += chunk_size - overlap

print("Total chunks created:", len(chunks))

if chunks:
    print("\nFirst chunk preview:\n")
    print(chunks[0]["text"])

Total chunks created: 54

First chunk preview:

802.11b/g/n Wireless USB Adapter 
 
1 
 
Content 
Chapter 1 Product Introduction .................................................................................................... 2 
1.1 Package Contents ................................................................................................................ 2 
1.2 Product Features .................................................................................................................. 2 
1.3 Product Usage .....................


In [10]:
from sentence_transformers import SentenceTransformer
import faiss
import numpy as np

# Load embedding model
embedding_model = SentenceTransformer(
    "sentence-transformers/all-MiniLM-L6-v2"
)

# Extract text from chunks
chunk_texts = [chunk["text"] for chunk in chunks]

# Generate embeddings
embeddings = embedding_model.encode(
    chunk_texts,
    convert_to_numpy=True,
    show_progress_bar=True
)

# Convert embeddings to float32
embeddings = np.asarray(embeddings, dtype="float32")

# Normalize embeddings for similarity search
faiss.normalize_L2(embeddings)

# Create FAISS vector database
dimension = embeddings.shape[1]
vector_db = faiss.IndexFlatIP(dimension)

# Add embeddings to database
vector_db.add(embeddings)

print("Total chunks:", len(chunks))
print("Embedding shape:", embeddings.shape)
print("FAISS database created successfully!")
print("Total vectors stored:", vector_db.ntotal)

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

...

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Batches:   0%|          | 0/2 [00:00<?, ?it/s]

Total chunks: 54
Embedding shape: (54, 384)
FAISS database created successfully!
Total vectors stored: 54


In [11]:
def retrieve_context(question, top_k=3):
    question_embedding = embedding_model.encode(
        [question],
        convert_to_numpy=True
    ).astype("float32")

    faiss.normalize_L2(question_embedding)

    k = min(top_k, vector_db.ntotal)
    scores, indices = vector_db.search(question_embedding, k)

    results = []

    for idx in indices[0]:
        if idx != -1:
            results.append(chunks[idx])

    return results


# Test retrieval
question = "What is this document about?"

results = retrieve_context(question)

for i, result in enumerate(results, start=1):
    print(f"\nResult {i}")
    print("Source:", result["source"])
    print("Page:", result["page"])
    print("Text:", result["text"][:500])


Result 1
Source: User Manual.pdf
Page: 17
Text: It is the six two-digit numbers that assemble the MAC address of respected joined station.  
6. Life Time  
It is the timer that counts down from 10 minutes whenever the AP connects the station 
successfully. If an STA associated to SW AP does not have any interaction with the AP in 
10 minutes, it will be disassociated from the Infra-structure BSS.  
7. Config  
A dialog of this AP is shown for configuration modification

Result 2
Source: User Manual.pdf
Page: 8
Text: tem. It is 
easy to use to switch property pages just by clicking left button of mouse on the title of 
each page. The following six sections describe detailed information of each page.  
A: General Page

Result 3
Source: User Manual.pdf
Page: 8
Text: 802.11b/g/n Wireless USB Adapter 
 
8 
 
Infrastructure and Ad-Hoc  
With both Infrastructure and Ad-Hoc types, the properties should look like the 
picture above. Six property pages present different information of current w

In [12]:
def rag_assistant(question, top_k=3):
    # Retrieve relevant information from the PDF
    results = retrieve_context(question, top_k=top_k)

    if not results:
        return "I could not find relevant information in the uploaded document."

    # Combine retrieved text
    context = "\n\n".join(
        result["text"] for result in results
    )

    # Create prompt
    prompt = f"""
You are a helpful personalized AI assistant.
Answer the user's question using the context below.
If the answer is not available in the context, clearly say so.
Do not invent facts.

Context:
{context}

Question:
{question}

Answer:
"""

    # Generate answer using your open-source LLM
    inputs = tokenizer(
        prompt,
        return_tensors="pt",
        truncation=True,
        max_length=2048
    ).to(model.device)

    outputs = model.generate(
        **inputs,
        max_new_tokens=250,
        do_sample=True,
        temperature=0.3,
        pad_token_id=tokenizer.eos_token_id
    )

    # Extract only newly generated answer
    new_tokens = outputs[0][inputs["input_ids"].shape[1]:]
    answer = tokenizer.decode(
        new_tokens,
        skip_special_tokens=True
    )

    return answer.strip()


print("RAG Assistant is ready!")

RAG Assistant is ready!


In [13]:
question = input("Ask a question about your PDF: ")

answer = rag_assistant(question)

print("\nAI Assistant Answer:")
print(answer)

Ask a question about your PDF: which driver 

AI Assistant Answer:
The 802.11b/g/n Wireless USB Adapter does not have an official driver installed by default. To resolve this issue, you need to download and install the appropriate driver from the manufacturer's website or through your device's support channels. Once the driver is installed, you can proceed with the setup process as described in the provided context. If you encounter any issues during the installation or setup, consult the manufacturer's documentation for further assistance. Additionally, ensure that your computer meets the minimum system requirements specified in the driver installation instructions. This will help prevent compatibility problems with other devices or operating systems. Remember, having an active driver installed ensures optimal performance and functionality of the wireless network adapter. If you're unsure about installing the driver, consider reaching out to a professional technician who specializes i

In [14]:
from IPython.display import display, Markdown
import ipywidgets as widgets

question_box = widgets.Text(
    placeholder="Ask something about your PDF...",
    description="Question:",
    layout=widgets.Layout(width="80%")
)

ask_button = widgets.Button(
    description="Ask AI",
    button_style="primary"
)

chat_output = widgets.Output()

def ask_question(b):
    question = question_box.value.strip()

    if not question:
        return

    with chat_output:
        display(Markdown(f"**You:** {question}"))

        try:
            answer = rag_assistant(question)
            display(Markdown(f"**AI Assistant:** {answer}"))
        except Exception as e:
            display(Markdown(f"**Error:** {str(e)}"))

    question_box.value = ""

ask_button.on_click(ask_question)

display(question_box, ask_button, chat_output)

Text(value='', description='Question:', layout=Layout(width='80%'), placeholder='Ask something about your PDF.…

Button(button_style='primary', description='Ask AI', style=ButtonStyle())

Output()

In [15]:
!pip install -q peft datasets

In [16]:
from datasets import Dataset

training_data = [
    {
        "question": "What is artificial intelligence?",
        "answer": "Artificial intelligence enables computers to perform tasks that usually require human intelligence."
    },
    {
        "question": "What is a large language model?",
        "answer": "A large language model is trained on text to understand and generate human-like language."
    },
    {
        "question": "What is retrieval augmented generation?",
        "answer": "RAG retrieves relevant information from external documents and uses it to generate grounded answers."
    },
    {
        "question": "What is fine-tuning?",
        "answer": "Fine-tuning further trains a pretrained model on task-specific examples."
    },
    {
        "question": "What is an AI assistant?",
        "answer": "An AI assistant helps users answer questions and complete tasks using artificial intelligence."
    }
]

dataset = Dataset.from_list(training_data)

print(dataset)
print("Training examples:", len(dataset))

Dataset({
    features: ['question', 'answer'],
    num_rows: 5
})
Training examples: 5


In [17]:
def tokenize_function(example):
    texts = [
        f"Question: {q}\nAnswer: {a}"
        for q, a in zip(
            example["question"],
            example["answer"]
        )
    ]

    return tokenizer(
        texts,
        truncation=True,
        max_length=256,
        padding="max_length"
    )

tokenized_dataset = dataset.map(
    tokenize_function,
    batched=True,
    remove_columns=dataset.column_names
)

print("Dataset tokenization completed!")

Map:   0%|          | 0/5 [00:00<?, ? examples/s]

Dataset tokenization completed!
